# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rupamsingh0907-beep/Flyintern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import numpy as np
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")
d = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset="content_id").copy()

d["is_down"] = (d["trend_direction"] == "down").astype(int)   # used ONLY to check signals, never as a rule input
print("n =", len(d), "| overall share trending down:", round(d["is_down"].mean(), 3))
print("missing days_since_last_update:", d["days_since_last_update"].isna().sum())

n = 30000 | overall share trending down: 0.542
missing days_since_last_update: 0


In [2]:
d["stale_bucket"] = pd.cut(d["days_since_last_update"], bins=[-1, 30, 90, 180, 365, np.inf],
                           labels=["0-30", "31-90", "91-180", "181-365", "365+"])
t1 = d.groupby("stale_bucket", observed=True)["is_down"].agg(n="size", share_down="mean").round(3)
print(t1)

                  n  share_down
stale_bucket                   
0-30          20480       0.511
31-90           175       0.589
91-180         9171       0.611
181-365         169       0.467
365+              5       0.600


In [3]:
d["vol_bucket"] = pd.qcut(d["impressions_90d"], q=4, duplicates="drop")
t2 = d.groupby("vol_bucket", observed=True)["is_down"].agg(n="size", share_down="mean").round(3)
print(t2)

                        n  share_down
vol_bucket                           
(0.999, 81.0]        7503       0.376
(81.0, 731.0]        7499       0.605
(731.0, 3615.25]     7498       0.626
(3615.25, 517715.0]  7500       0.562


**Rule idea:** send pages that are stale and visible to refresh review. Staleness is the signal behind FlyRank's refresh flags, and volume is the signal behind quick-win. I check both against whether a page is currently trending down (a proxy used for checking only, never as a rule input).

**Signal 1: staleness.** Hypothesis: pages not updated for longer are more likely to be trending down. The table above shows the share trending down by days since last update, with n in each bucket. It goes from [X] in the freshest bucket to [Y] in the stalest. **Verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE].** [One or two sentences on why, and what it means for the rule.]

**Signal 2: volume.** Hypothesis: higher-volume pages are more likely to be trending down. The share goes from [X] in the lowest quartile to [Y] in the highest. **Verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE].** [If this is FALSE or MIXED, say so plainly: volume then works as a measure of how much is at stake, not as evidence of decline. That is a clear negative and it sharpens the rule.]

The overall share trending down is [Z], so a bucket only matters if it differs clearly from that. Both checks are associations in the current window, not proof that staleness causes decline.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
elig = d[(d["days_since_last_update"] >= 180) & (d["impressions_90d"] >= 500)].copy()

elig["stale_pct"] = elig["days_since_last_update"].rank(pct=True)
elig["vis_pct"]   = elig["impressions_90d"].rank(pct=True)
elig["score"]       = (elig["stale_pct"] * elig["vis_pct"]).round(4)
elig["reason_code"] = "stale_visible_page"
elig["action"]      = "refresh_review"

queue = elig.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

out_cols = ["rank", "content_id", "client_id", "score", "reason_code", "action",
            "days_since_last_update", "impressions_90d"]
os.makedirs("work/outputs", exist_ok=True)
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(len(queue), "pages in the queue")
queue[out_cols].head(10)

17 pages in the queue


,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_cf56e2e2e282,client_7f2253d7e2,0.6471,stale_visible_page,refresh_review,194,61678
1,2,content_7368877ea310,client_7f2253d7e2,0.6090,stale_visible_page,refresh_review,194,59472
2,3,content_1bfaa38ff26c,client_7f2253d7e2,0.5709,stale_visible_page,refresh_review,194,25715
3,4,content_5feee3994adb,client_7f2253d7e2,0.4948,stale_visible_page,refresh_review,194,7812
4,5,content_b16bd7307b39,client_7f2253d7e2,0.4187,stale_visible_page,refresh_review,194,4590
5,6,content_fe16a55cd13d,client_7f2253d7e2,0.3806,stale_visible_page,refresh_review,194,4556
6,7,content_ecb6215e79fd,client_7f2253d7e2,0.3426,stale_visible_page,refresh_review,194,4429
7,8,content_7f116ae1f6f5,client_9400f1b21c,0.2855,stale_visible_page,refresh_review,301,954
8,9,content_0a91db491d14,client_7f2253d7e2,0.2422,stale_visible_page,refresh_review,193,13299
9,10,content_bdbec75c1148,client_7f2253d7e2,0.2284,stale_visible_page,refresh_review,194,1316


**Rule:** a page is eligible if it has gone 180+ days without an update and has 500+ impressions in 90 days. Its score is its percentile rank for staleness times its percentile rank for impressions, so the highest scores are pages that are both very stale and very visible.

- **Score:** `stale_pct x vis_pct`, from 0 to 1.
- **Reason code:** `stale_visible_page` (the only one).
- **Action label:** `refresh_review`, meaning a person looks at the page and decides whether to refresh it.

Inputs are only `days_since_last_update` and `impressions_90d`. No trend columns and no future-window values are used. The ranked queue is written to `work/outputs/baseline_action_score.csv`.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
ctx = ["content_id", "score", "days_since_last_update", "impressions_90d",
       "content_type", "avg_position", "ctr", "word_count", "trend_direction"]
queue[ctx].head(10)

,content_id,score,days_since_last_update,impressions_90d,content_type,avg_position,ctr,word_count,trend_direction
0,content_cf56e2e2e282,0.6471,194,61678,keyword article,19.7,0.15,5125.0,down
1,content_7368877ea310,0.6090,194,59472,keyword article,24.8,0.13,2591.0,down
2,content_1bfaa38ff26c,0.5709,194,25715,keyword article,22.2,0.23,3861.0,down
3,content_5feee3994adb,0.4948,194,7812,keyword article,39.0,0.01,3590.0,down
4,content_b16bd7307b39,0.4187,194,4590,keyword article,31.0,0.00,4329.0,down
5,content_fe16a55cd13d,0.3806,194,4556,keyword article,16.4,0.33,3388.0,down
6,content_ecb6215e79fd,0.3426,194,4429,keyword article,25.3,0.38,4486.0,down
7,content_7f116ae1f6f5,0.2855,301,954,keyword article,9.0,0.42,1335.0,down
8,content_0a91db491d14,0.2422,193,13299,keyword article,10.5,0.49,3478.0,down
9,content_bdbec75c1148,0.2284,194,1316,keyword article,21.8,0.15,3696.0,stable


1. **`[content_id]`: refresh_review.** Why: [N] days since update with [N] impressions in 90 days. What would make it wrong: [your reason].
2. ...
(repeat for all ten)

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weakest of my top ten are rows [A], [B] and [C]. [A] is weak because [reason, for example it is already trending up]. [B] is weak because [reason]. [C] is weak because [reason]. This suggests the rule is a rough first filter, not a verdict, and a reviewer should check these before acting.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.